In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
# 개별 json 라벨 파일을 이용해 학습 데이터 리스트 생성
import glob
import json
import os
import pandas as pd
from PIL import Image
from torch.utils import data
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:",device)

In [ ]:
# M1 TCGA-only 데이터 로드
# 입력 단위: 1 slide(case)의 tile image paths + tile 좌표 + slide 전체 크기 + multi-horizon survival label/mask
# 출력 label: dead by 6/12/18/24 months. Unknown(censored before horizon)은 mask=0으로 loss에서 제외합니다.

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"

M1_OUTPUT_PATH = Path("outputs/M1")
M1_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "TCGA_PAAD"
SEED = 42
PATCH_INPUT_SIZE = 512  # 1.0 MPP 1024 tile을 512 입력으로 줄이면 effective MPP는 약 2.0입니다.
PRELOAD_RESIZED_TILES = True  # dataset 구성 단계에서 512x512 tile을 CPU memory에 preload합니다.
PRELOAD_TILE_SIZE = PATCH_INPUT_SIZE
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)
TEST_SIZE = 0.2
VALID_SIZE = 0.25  # train_valid 내부 비율. 전체 기준 0.8 * 0.25 = 0.2
REQUIRE_COMPLETE_24M_HORIZONS = True

MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_clinical_label(dataset: str, case_id: str) -> tuple[float | None, int | None]:
    clinical_json_path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not clinical_json_path.exists():
        return None, None
    with open(clinical_json_path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    clinical = clinical_json.get("clinical", {})
    return clinical.get("os_time_days"), clinical.get("os_event")


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    """Create multi-horizon dead-by labels and known-label mask.

    label[h] = 1 if death occurred by horizon h, else 0.
    mask[h] = 1 if the label at horizon h is known.

    Censored before a horizon is unknown and excluded from BCE loss with mask=0.
    """
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask

    os_time_days = float(os_time_days)
    os_event = int(os_event)

    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            # Censored before this horizon: unknown.
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def get_patch_padding(image_size: int = PATCH_INPUT_SIZE) -> tuple[int, int, int, int]:
    # UNI2-h는 patch_size=14라 512 입력이 바로 들어갈 수 없습니다.
    # 512로 먼저 resize한 뒤 가장 가까운 patch-size multiple까지 symmetric padding합니다.
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    target_size = int(np.ceil(image_size / patch_size) * patch_size)
    pad_total = max(0, target_size - image_size)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return (pad_left, pad_left, pad_right, pad_right)


def get_model_input_size(image_size: int = PATCH_INPUT_SIZE) -> int:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    return int(np.ceil(image_size / patch_size) * patch_size)


def get_train_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    # Resize를 augmentation보다 먼저 수행해 1024 tile에서 바로 연산하지 않도록 합니다.
    # 이후 patch-size multiple로 padding합니다. UNI2-h는 512 -> 518 padding이 필요합니다.
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_train_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    # dataset 구성 단계에서 이미 512x512로 resize된 image에 대해 padding/augmentation/normalization만 수행합니다.
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def estimate_tile_cache_gb(n_tiles: int, image_size: int = PRELOAD_TILE_SIZE) -> float:
    return n_tiles * image_size * image_size * 3 / (1024 ** 3)


def preload_resized_tile_images(tile_index: pd.DataFrame, image_size: int = PRELOAD_TILE_SIZE) -> dict[str, np.ndarray]:
    unique_paths = sorted(tile_index["tile_path"].astype(str).unique())
    expected_gb = estimate_tile_cache_gb(len(unique_paths), image_size=image_size)
    print(f"Preloading resized tiles: {len(unique_paths):,} tiles, {image_size}x{image_size}, expected uint8 memory ~{expected_gb:.2f} GB")

    cache = {}
    resize_size = (image_size, image_size)
    for tile_path in tqdm(unique_paths, desc="Preload resized tile images", unit="tile"):
        with Image.open(tile_path) as image:
            image = image.convert("RGB")
            image = image.resize(resize_size, resample=Image.BILINEAR)
            cache[tile_path] = np.asarray(image, dtype=np.uint8).copy()
    return cache


def load_tcga_case_tiles(summary_row: pd.Series) -> tuple[pd.DataFrame, dict]:
    case_id = str(summary_row["case_id"])
    case_dir = Path(summary_row["case_dir"])
    metadata_path = Path(summary_row["metadata_path"])
    tile_df = pd.read_csv(metadata_path)

    tile_df["x"] = tile_df["x_level0"].astype(int)
    tile_df["y"] = tile_df["y_level0"].astype(int)
    slide_width = int(summary_row["slide_width"])
    slide_height = int(summary_row["slide_height"])

    tile_df["tile_path"] = tile_df["tile_path"].astype(str)
    tile_df = tile_df[tile_df["tile_path"].map(lambda x: Path(x).exists())].copy()
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height

    source_size = tile_df["source_tile_size_px"].astype(float)
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height

    os_time, os_event = load_clinical_label(DATASET_NAME, case_id)
    if os_time is None or os_event is None:
        os_time = tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan
        os_event = tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan

    y, mask = make_horizon_label_mask(os_time, os_event)
    case_record = {
        "dataset": DATASET_NAME,
        "case_id": case_id,
        "case_dir": case_dir.as_posix(),
        "metadata_path": metadata_path.as_posix(),
        "n_tiles": int(len(tile_df)),
        "slide_width": slide_width,
        "slide_height": slide_height,
        "os_time_days": float(os_time) if pd.notna(os_time) else np.nan,
        "os_event": int(os_event) if pd.notna(os_event) else np.nan,
        "known_horizon_count": int(mask.sum()),
    }
    for i, name in enumerate(HORIZON_NAMES):
        case_record[name] = float(y[i])
        case_record[f"mask_{name}"] = float(mask[i])
    return tile_df, case_record


summary_path = IMAGE_PATH / DATASET_NAME / "tile_summary.csv"
summary_df = pd.read_csv(summary_path)
summary_df = summary_df[summary_df["metadata_path"].notna()].copy()

case_records = []
tile_index_list = []

for _, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc=f"Load {DATASET_NAME} metadata"):
    metadata_path = Path(row["metadata_path"])
    if not metadata_path.exists():
        continue
    tile_df, case_record = load_tcga_case_tiles(row)
    if case_record["n_tiles"] == 0:
        case_record["exclude_reason"] = "no_tiles"
        case_records.append(case_record)
        continue
    tile_df["dataset"] = DATASET_NAME
    tile_df["case_id"] = case_record["case_id"]
    tile_index_list.append(tile_df)
    case_records.append(case_record)

all_slide_df = pd.DataFrame(case_records)
tile_index_df = pd.concat(tile_index_list, ignore_index=True)

complete_horizon_mask = all_slide_df[[f"mask_{name}" for name in HORIZON_NAMES]].eq(1).all(axis=1)
required_horizon_mask = complete_horizon_mask if REQUIRE_COMPLETE_24M_HORIZONS else all_slide_df["known_horizon_count"].gt(0)

eligible_mask = (
    all_slide_df["os_time_days"].notna()
    & all_slide_df["os_event"].notna()
    & required_horizon_mask
    & all_slide_df["n_tiles"].gt(0)
)
excluded_df = all_slide_df[~eligible_mask].copy()
if not excluded_df.empty:
    excluded_df["exclude_reason"] = np.select(
        [
            excluded_df["n_tiles"].le(0),
            excluded_df["os_time_days"].isna(),
            excluded_df["os_event"].isna(),
            ~required_horizon_mask.loc[excluded_df.index],
        ],
        ["no_tiles", "missing_os_time", "missing_os_event", "incomplete_required_horizon"],
        default="unknown",
    )

slide_df = all_slide_df[eligible_mask].copy()
slide_df["os_event"] = slide_df["os_event"].astype(int)
tile_index_df = tile_index_df[
    tile_index_df["case_id"].isin(slide_df["case_id"])
    & tile_index_df["dataset"].eq(DATASET_NAME)
].copy()

slide_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_slide_manifest.csv", index=False)
tile_index_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_tile_index.csv", index=False)
excluded_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_excluded_cases.csv", index=False)

print("all_slide_df:", all_slide_df.shape)
print("complete 24m horizon cases:", int(complete_horizon_mask.sum()))
print("slide_df:", slide_df.shape)
print("excluded_df:", excluded_df.shape)
print("tile_index_df:", tile_index_df.shape)

if PRELOAD_RESIZED_TILES:
    if "TILE_IMAGE_CACHE" not in globals() or not TILE_IMAGE_CACHE:
        TILE_IMAGE_CACHE = preload_resized_tile_images(tile_index_df, image_size=PRELOAD_TILE_SIZE)
    else:
        print(f"Using existing TILE_IMAGE_CACHE: {len(TILE_IMAGE_CACHE):,} tiles")
else:
    TILE_IMAGE_CACHE = {}

print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("cached tiles:", len(TILE_IMAGE_CACHE))

horizon_summary = []
for name in HORIZON_NAMES:
    mask_col = f"mask_{name}"
    known = slide_df[mask_col].eq(1)
    horizon_summary.append({
        "horizon": name,
        "known_n": int(known.sum()),
        "dead_n": int(slide_df.loc[known, name].sum()),
        "alive_n": int(known.sum() - slide_df.loc[known, name].sum()),
        "unknown_n": int((~known).sum()),
        "dead_rate_known": float(slide_df.loc[known, name].mean()) if known.any() else np.nan,
    })
horizon_summary_df = pd.DataFrame(horizon_summary)
display(horizon_summary_df)
display(slide_df[["case_id", "n_tiles", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].head())


class M1SlideDataset(Dataset):
    """TCGA case-level pathology dataset for multi-horizon MIL training."""

    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            case_id: group.sort_values(["y", "x"]).reset_index(drop=True)
            for case_id, group in tile_index.groupby("case_id")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["case_id"]]

        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        label = row[HORIZON_NAMES].to_numpy(np.float32)
        mask = row[[f"mask_{name}" for name in HORIZON_NAMES]].to_numpy(np.float32)
        slide_size = np.array([row["slide_width"], row["slide_height"]], dtype=np.float32)

        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "slide_size": torch.from_numpy(slide_size),
            "horizon_months": torch.tensor(HORIZON_MONTHS, dtype=torch.float32),
            "label": torch.from_numpy(label),
            "mask": torch.from_numpy(mask),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
        }


class M1TileDataset(Dataset):
    """Tile-level dataset for UNI/UNI v2 on-the-fly feature extraction with augmentation."""

    def __init__(self, tile_index: pd.DataFrame, transform=None, tile_cache: dict[str, np.ndarray] | None = None):
        self.tile_index = tile_index.reset_index(drop=True).copy()
        self.transform = transform
        self.tile_cache = tile_cache or {}

    def __len__(self):
        return len(self.tile_index)

    def __getitem__(self, idx):
        row = self.tile_index.iloc[idx]
        tile_path = row["tile_path"]
        if tile_path in self.tile_cache:
            image = Image.fromarray(self.tile_cache[tile_path])
        else:
            with Image.open(tile_path) as image_file:
                image = image_file.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coords = row[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {
            "image": image,
            "coords": torch.from_numpy(coords),
            "case_id": row["case_id"],
            "tile_path": tile_path,
        }


# 24개월까지 모든 horizon label이 확인 가능한 TCGA case만 사용하고, 전체 기준 6:2:2로 split합니다.
train_valid_df, test_df = train_test_split(
    slide_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=slide_df["os_event"],
)
train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=train_valid_df["os_event"],
)

train_case_ids = set(train_df["case_id"])
valid_case_ids = set(valid_df["case_id"])
test_case_ids = set(test_df["case_id"])
assert len(train_case_ids & valid_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(valid_case_ids & test_case_ids) == 0

train_dataset = M1SlideDataset(train_df, tile_index_df)
valid_dataset = M1SlideDataset(valid_df, tile_index_df)
test_dataset = M1SlideDataset(test_df, tile_index_df)

tile_train_transform = get_train_cached_patch_transform() if TILE_IMAGE_CACHE else get_train_patch_transform()
tile_eval_transform = get_eval_cached_patch_transform() if TILE_IMAGE_CACHE else get_eval_patch_transform()

train_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["case_id"].isin(train_case_ids)],
    transform=tile_train_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
valid_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["case_id"].isin(valid_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
test_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["case_id"].isin(test_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)

split_df = slide_df[["dataset", "case_id", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].copy()
split_df["split"] = "unused"
split_df.loc[split_df["case_id"].isin(train_case_ids), "split"] = "train"
split_df.loc[split_df["case_id"].isin(valid_case_ids), "split"] = "valid"
split_df.loc[split_df["case_id"].isin(test_case_ids), "split"] = "test"
split_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_horizon_case_splits.csv", index=False)

print("slide splits:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("tile splits:", len(train_tile_dataset), len(valid_tile_dataset), len(test_tile_dataset))
display(pd.crosstab(split_df["split"], split_df["os_event"]))

split_horizon_summary = []
for split_name, part in split_df.groupby("split"):
    row = {"split": split_name, "n_cases": len(part)}
    for name in HORIZON_NAMES:
        known = part[f"mask_{name}"].eq(1)
        row[f"{name}_known"] = int(known.sum())
        row[f"{name}_dead"] = int(part.loc[known, name].sum())
    split_horizon_summary.append(row)
display(pd.DataFrame(split_horizon_summary))

sample = train_dataset[0]
print("sample case:", sample["case_id"])
print("n_tiles:", len(sample["tile_paths"]))
print("coords:", sample["coords"].shape, sample["coords"][:3])
print("slide_size:", sample["slide_size"])
print("label:", sample["label"])
print("mask:", sample["mask"])
print("model input size:", get_model_input_size(PATCH_INPUT_SIZE), "padding:", get_patch_padding(PATCH_INPUT_SIZE))
print("model output 해석: logits -> sigmoid(logits) * 100 = horizon별 사망위험 percent")


In [ ]:
# M1 모델 정의 및 loss/optimizer 구성
# 구조: frozen UNI/UNI v2 feature extractor + coordinate spatial embedding + attention MIL

import torch
from torch import nn
import pandas as pd
import timm
from huggingface_hub import hf_hub_download

from scripts.models.discrete_survival import (
    cox_ph_loss,
    harrell_c_index,
    horizon_auc_metrics,
)

from scripts.models.m1_pathology_mil import (
    M1ModelConfig,
    PathologySpatialMIL,
    sample_tiles,
    build_optimizer,
)

if "device" not in globals():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

MAX_TILES_PER_SLIDE = 10_000
FEATURE_BATCH_SIZE = 64
SPATIAL_DIM = 128
FUSION_DIM = 256
MIL_HIDDEN_DIM = 128
USE_SPATIAL_EMBEDDING = False
POOLING_MODE = "mean"
DROPOUT = 0.40
N_OUTPUTS = 1  # single Cox risk score
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# "UNI" 또는 "UNI2-h" 중 선택하세요.
# UNI/UNI2-h는 Hugging Face 접근 권한 또는 로컬 캐시가 필요할 수 있습니다.
FEATURE_EXTRACTOR_NAME = "UNI2-h"
UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}


def load_uni_feature_extractor(
    name: str = FEATURE_EXTRACTOR_NAME,
    device: torch.device = device,
    local_files_only: bool = False,
) -> tuple[nn.Module, int]:
    if name not in UNI_BACKBONES:
        raise ValueError(f"지원하지 않는 feature extractor입니다: {name}. choices={list(UNI_BACKBONES)}")

    cfg = UNI_BACKBONES[name]
    print(f"Loading {name}: {cfg['repo_id']} / {cfg['timm_kwargs']['model_name']}")

    model = timm.create_model(
        pretrained=False,
        **cfg["timm_kwargs"],
    )

    try:
        weight_path = hf_hub_download(
            repo_id=cfg["repo_id"],
            filename=cfg["filename"],
            local_files_only=local_files_only,
        )
    except Exception as exc:
        raise RuntimeError(
            f"{name} weight를 Hugging Face에서 가져오지 못했습니다. "
            "접근 권한/로그인 토큰 또는 네트워크/캐시를 확인하세요. "
            f"repo_id={cfg['repo_id']}, filename={cfg['filename']}"
        ) from exc

    state_dict = torch.load(weight_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("load_state_dict warning")
        print("missing keys:", len(missing))
        print("unexpected keys:", len(unexpected))

    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device)

    feature_dim = int(getattr(model, "num_features", cfg["feature_dim"]))
    print(f"{name} loaded. feature_dim={feature_dim}")
    return model, feature_dim


# 셀 3에서 UNI/UNI v2까지 로드하고 바로 M1 model/optimizer를 구성합니다.
feature_extractor, M1_FEATURE_DIM = load_uni_feature_extractor(
    name=FEATURE_EXTRACTOR_NAME,
    device=device,
    local_files_only=False,
)
FEATURE_EXTRACTOR_PATCH_SIZE = int(UNI_BACKBONES[FEATURE_EXTRACTOR_NAME]["timm_kwargs"]["patch_size"])
print("FEATURE_EXTRACTOR_PATCH_SIZE:", FEATURE_EXTRACTOR_PATCH_SIZE)
print("PATCH_INPUT_SIZE:", PATCH_INPUT_SIZE, "-> model input size:", get_model_input_size(PATCH_INPUT_SIZE))
print("PATCH_PADDING:", get_patch_padding(PATCH_INPUT_SIZE))

# 2번 셀에서 만든 tile-level dataset transform은 feature extractor patch size를 알기 전 생성되므로 여기서 갱신합니다.
# resized tile cache가 있으면 512 resize를 반복하지 않고 padding/augmentation/normalization만 수행합니다.
if bool(globals().get("TILE_IMAGE_CACHE", {})):
    train_tile_dataset.transform = get_train_cached_patch_transform()
    valid_tile_dataset.transform = get_eval_cached_patch_transform()
    test_tile_dataset.transform = get_eval_cached_patch_transform()
else:
    train_tile_dataset.transform = get_train_patch_transform()
    valid_tile_dataset.transform = get_eval_patch_transform()
    test_tile_dataset.transform = get_eval_patch_transform()

m1_config = M1ModelConfig(
    feature_dim=M1_FEATURE_DIM,
    coord_dim=6,
    spatial_dim=SPATIAL_DIM,
    fusion_dim=FUSION_DIM,
    mil_hidden_dim=MIL_HIDDEN_DIM,
    n_outputs=N_OUTPUTS,
    dropout=DROPOUT,
    max_tiles=MAX_TILES_PER_SLIDE,
    feature_batch_size=FEATURE_BATCH_SIZE,
    freeze_feature_extractor=True,
    use_spatial_embedding=USE_SPATIAL_EMBEDDING,
    pooling_mode=POOLING_MODE,
)


def m1_loss_fn(logits: torch.Tensor, os_time_days: torch.Tensor, os_event: torch.Tensor) -> torch.Tensor:
    return cox_ph_loss(
        risk_scores=logits.float().reshape(-1),
        times=os_time_days,
        events=os_event,
    )


def initialize_m1_model(
    feature_extractor: nn.Module,
    feature_dim: int = M1_FEATURE_DIM,
    lr: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
) -> tuple[PathologySpatialMIL, torch.optim.Optimizer]:
    config = M1ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
        use_spatial_embedding=USE_SPATIAL_EMBEDDING,
        pooling_mode=POOLING_MODE,
    )
    model = PathologySpatialMIL(feature_extractor=feature_extractor, config=config).to(device)
    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay)
    return model, optimizer


model, optimizer = initialize_m1_model(feature_extractor, feature_dim=M1_FEATURE_DIM)

print("M1 model initialized.")
print("FEATURE_EXTRACTOR_NAME:", FEATURE_EXTRACTOR_NAME)
print("M1_FEATURE_DIM:", M1_FEATURE_DIM)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("MAX_TILES_PER_SLIDE:", MAX_TILES_PER_SLIDE)
print("FEATURE_BATCH_SIZE:", FEATURE_BATCH_SIZE)
print("USE_SPATIAL_EMBEDDING:", USE_SPATIAL_EMBEDDING)
print("POOLING_MODE:", POOLING_MODE)
print("N_OUTPUTS:", N_OUTPUTS, HORIZON_NAMES)

# train loop에서 사용할 loss 예시:
# outputs = model(tile_images, coords)
# loss = m1_loss_fn(outputs["logits"], labels, masks)
# hazard_percent = outputs["hazard_percent"]
# cumulative_risk_percent = outputs["risk_percent"]


In [ ]:
# M1 학습 하이퍼파라미터 정의

from pathlib import Path
import time
import json

MODEL_PATH = Path("../../model")
M1_MODEL_DIR = MODEL_PATH / "pancreatic_cancer_pathology" / "M1" / FEATURE_EXTRACTOR_NAME
M1_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
PATIENCE = 15
MIN_DELTA = 1e-4
MONITOR_METRIC = "valid_mean_auroc"
MONITOR_MODE = "max"
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

# 현재 구조는 slide 1개씩 학습합니다.
SLIDE_BATCH_SIZE = 1
LOG_EVERY_EPOCHS = 10
SAVE_EVERY_EPOCHS = 10
CASE_BATCH_SIZE = 8


BEST_CHECKPOINT_PATH = M1_MODEL_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = M1_MODEL_DIR / "last_checkpoint.pt"
TRAIN_LOG_PATH = M1_MODEL_DIR / "training_log.csv"
CONFIG_PATH = M1_MODEL_DIR / "training_config.json"

training_config = {
    "dataset": DATASET_NAME,
    "feature_extractor": FEATURE_EXTRACTOR_NAME,
    "patch_input_size": PATCH_INPUT_SIZE,
    "model_input_size": get_model_input_size(PATCH_INPUT_SIZE),
    "preload_resized_tiles": PRELOAD_RESIZED_TILES,
    "preload_tile_size": PRELOAD_TILE_SIZE,
    "patch_padding": get_patch_padding(PATCH_INPUT_SIZE),
    "feature_extractor_patch_size": FEATURE_EXTRACTOR_PATCH_SIZE,
    "effective_mpp": 1.0 * 1024 / PATCH_INPUT_SIZE,
    "max_tiles_per_slide": MAX_TILES_PER_SLIDE,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "horizon_months": HORIZON_MONTHS,
    "horizon_names": HORIZON_NAMES,
    "loss_function": "cox_partial_likelihood",
    "output_interpretation": "single_risk_score",
    "case_batch_size": CASE_BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "monitor_metric": MONITOR_METRIC,
    "monitor_mode": MONITOR_MODE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "spatial_dim": SPATIAL_DIM,
    "fusion_dim": FUSION_DIM,
    "mil_hidden_dim": MIL_HIDDEN_DIM,
    "use_spatial_embedding": USE_SPATIAL_EMBEDDING,
    "pooling_mode": POOLING_MODE,
    "use_amp": USE_AMP,
    "amp_dtype": str(AMP_DTYPE),
    "seed": SEED,
    "model_dir": M1_MODEL_DIR.as_posix(),
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

# Scheduler는 validation loss 기준으로 learning rate를 줄입니다.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=MONITOR_MODE,
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

print("M1_MODEL_DIR:", M1_MODEL_DIR)
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("EPOCHS:", EPOCHS)
print("PATIENCE:", PATIENCE)
print("CASE_BATCH_SIZE:", CASE_BATCH_SIZE)
print("USE_AMP:", USE_AMP)
print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("scheduler:", scheduler.__class__.__name__)
print("MONITOR_METRIC:", MONITOR_METRIC, MONITOR_MODE)


In [ ]:
# M1 학습 및 validation
# single-risk Cox survival loss, C-index, horizon-wise AUROC/AUPRC, checkpoint, scheduler

from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def load_tile_tensor_batch(tile_paths: list[str], transform, device: torch.device) -> torch.Tensor:
    images = []
    use_cache = bool(globals().get("TILE_IMAGE_CACHE", {}))
    for path in tile_paths:
        if use_cache and path in TILE_IMAGE_CACHE:
            image = Image.fromarray(TILE_IMAGE_CACHE[path])
        else:
            with Image.open(path) as image_file:
                image = image_file.convert("RGB")
        images.append(transform(image))
    return torch.stack(images, dim=0).to(device, non_blocking=True)


def prepare_slide_batch(sample: dict, training: bool) -> dict:
    if bool(globals().get("TILE_IMAGE_CACHE", {})):
        transform = get_train_cached_patch_transform() if training else get_eval_cached_patch_transform()
    else:
        transform = get_train_patch_transform() if training else get_eval_patch_transform()
    selected_paths, selected_coords, selected_indices = sample_tiles(
        sample["tile_paths"],
        sample["coords"],
        max_tiles=MAX_TILES_PER_SLIDE,
        training=training,
    )
    tile_images = load_tile_tensor_batch(selected_paths, transform=transform, device=device)
    return {
        "tile_images": tile_images,
        "coords": selected_coords.to(device, non_blocking=True),
        "labels": sample["label"].unsqueeze(0).to(device, non_blocking=True).float(),
        "masks": sample["mask"].unsqueeze(0).to(device, non_blocking=True).float(),
        "os_time_days": sample["os_time_days"].reshape(1).to(device, non_blocking=True).float(),
        "os_event": sample["os_event"].reshape(1).to(device, non_blocking=True).long(),
        "case_id": sample["case_id"],
        "n_tiles": len(selected_paths),
    }


def compute_epoch_metrics(risk_scores: list[float], times: list[float], events: list[int], labels: list[np.ndarray], masks: list[np.ndarray]) -> dict:
    risk_array = np.asarray(risk_scores, dtype=float)
    label_array = np.vstack(labels) if labels else np.empty((0, len(HORIZON_NAMES)))
    mask_array = np.vstack(masks) if masks else np.empty((0, len(HORIZON_NAMES)))
    risk_matrix = np.repeat(risk_array[:, None], len(HORIZON_NAMES), axis=1) if len(risk_array) else np.empty((0, len(HORIZON_NAMES)))
    metrics = {
        "c_index": harrell_c_index(times, events, risk_array) if len(risk_array) else float("nan"),
    }
    metrics.update(horizon_auc_metrics(label_array, mask_array, risk_matrix, HORIZON_NAMES) if len(risk_array) else {})
    return metrics


def run_one_epoch(dataset, training: bool, epoch: int) -> dict:
    if training:
        model.train()
        model.feature_extractor.eval()
    else:
        model.eval()

    total_loss = 0.0
    total_batches = 0
    all_times = []
    all_events = []
    all_risks = []
    all_labels = []
    all_masks = []

    indices = list(range(len(dataset)))
    if training:
        random.shuffle(indices)

    desc = f"Epoch {epoch:03d} train" if training else f"Epoch {epoch:03d} valid"
    pbar = tqdm(range(0, len(indices), CASE_BATCH_SIZE), desc=desc, leave=False)

    for start in pbar:
        batch_indices = indices[start:start + CASE_BATCH_SIZE]
        logits_list = []
        times_list = []
        events_list = []
        batch_tile_count = 0

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            for idx in batch_indices:
                sample = dataset[idx]
                batch = prepare_slide_batch(sample, training=training)
                if USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        outputs = model(batch["tile_images"], batch["coords"])
                else:
                    outputs = model(batch["tile_images"], batch["coords"])

                logits_list.append(outputs["logits"].reshape(-1))
                times_list.append(batch["os_time_days"].reshape(-1))
                events_list.append(batch["os_event"].reshape(-1))
                batch_tile_count += batch["n_tiles"]

                with torch.no_grad():
                    all_risks.append(float(outputs["logits"].detach().reshape(-1)[0].cpu()))
                    all_times.extend(batch["os_time_days"].detach().cpu().numpy().tolist())
                    all_events.extend(batch["os_event"].detach().cpu().numpy().astype(int).tolist())
                    all_labels.append(batch["labels"].detach().cpu().numpy()[0])
                    all_masks.append(batch["masks"].detach().cpu().numpy()[0])

                del batch, outputs

            logits_tensor = torch.cat(logits_list, dim=0)
            times_tensor = torch.cat(times_list, dim=0)
            events_tensor = torch.cat(events_list, dim=0)
            loss = m1_loss_fn(logits_tensor, times_tensor, events_tensor)

            if training:
                loss.backward()
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad],
                        GRAD_CLIP_NORM,
                    )
                optimizer.step()

        total_loss += float(loss.detach().cpu())
        total_batches += 1
        running_loss = total_loss / max(total_batches, 1)
        running_metrics = compute_epoch_metrics(all_risks, all_times, all_events, all_labels, all_masks)
        pbar.set_postfix({
            "avg_loss": f"{running_loss:.4f}",
            "c_index": "nan" if np.isnan(running_metrics["c_index"]) else f"{running_metrics['c_index']:.3f}",
            "last_loss": f"{float(loss.detach().cpu()):.4f}",
            "tiles": batch_tile_count,
        })

        del logits_list, times_list, events_list, logits_tensor, times_tensor, events_tensor, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metrics = {"loss": total_loss / max(total_batches, 1)}
    metrics.update(compute_epoch_metrics(all_risks, all_times, all_events, all_labels, all_masks))
    return metrics


def get_mil_state_dict(model: torch.nn.Module) -> dict:
    return {key: value.detach().cpu() for key, value in model.state_dict().items() if not key.startswith("feature_extractor.")}


def save_checkpoint(path: Path, epoch: int, best_monitor_score: float, history: list[dict]):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": get_mil_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_monitor_score": best_monitor_score,
        "history": history,
        "config": training_config,
        "horizon_names": HORIZON_NAMES,
        "loss_function": "cox_partial_likelihood",
        "output_interpretation": "single_risk_score",
        "note": "feature_extractor weights are excluded; reload UNI/UNI2-h before loading this checkpoint with strict=False.",
    }
    torch.save(checkpoint, path)


def plot_training_history(history: list[dict]):
    hist_df = pd.DataFrame(history)
    display(hist_df.tail(10))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0, 0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
    axes[0, 0].plot(hist_df["epoch"], hist_df["valid_loss"], label="valid")
    axes[0, 0].set_title("Cox Loss")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].legend()

    axes[0, 1].plot(hist_df["epoch"], hist_df["train_c_index"], label="train")
    axes[0, 1].plot(hist_df["epoch"], hist_df["valid_c_index"], label="valid")
    axes[0, 1].set_title("C-index")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].legend()

    axes[1, 0].plot(hist_df["epoch"], hist_df["train_mean_auroc"], label="train AUROC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["valid_mean_auroc"], label="valid AUROC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["train_mean_auprc"], linestyle="--", label="train AUPRC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["valid_mean_auprc"], linestyle="--", label="valid AUPRC")
    axes[1, 0].set_title("Mean Horizon AUROC/AUPRC")
    axes[1, 0].set_xlabel("epoch")
    axes[1, 0].legend()

    axes[1, 1].plot(hist_df["epoch"], hist_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("epoch")
    axes[1, 1].set_yscale("log")
    axes[1, 1].legend()
    plt.tight_layout()
    plt.show()


history = []
best_monitor_score = -float("inf") if MONITOR_MODE == "max" else float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch(train_dataset, training=True, epoch=epoch)
    valid_metrics = run_one_epoch(valid_dataset, training=False, epoch=epoch)
    current_lr = optimizer.param_groups[0]["lr"]

    log_row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "valid_loss": valid_metrics["loss"],
        "train_c_index": train_metrics["c_index"],
        "valid_c_index": valid_metrics["c_index"],
        "train_mean_auroc": train_metrics.get("mean_auroc", float("nan")),
        "valid_mean_auroc": valid_metrics.get("mean_auroc", float("nan")),
        "train_mean_auprc": train_metrics.get("mean_auprc", float("nan")),
        "valid_mean_auprc": valid_metrics.get("mean_auprc", float("nan")),
        "lr": current_lr,
    }
    for name in HORIZON_NAMES:
        for metric_name in ["known", "positive", "auroc", "auprc"]:
            key = f"{name}_{metric_name}"
            log_row[f"train_{key}"] = train_metrics.get(key, float("nan"))
            log_row[f"valid_{key}"] = valid_metrics.get(key, float("nan"))

    monitor_score = log_row[MONITOR_METRIC]
    scheduler.step(monitor_score)
    history.append(log_row)
    pd.DataFrame(history).to_csv(TRAIN_LOG_PATH, index=False)

    if MONITOR_MODE == "max":
        improved = monitor_score > (best_monitor_score + MIN_DELTA)
    else:
        improved = monitor_score < (best_monitor_score - MIN_DELTA)

    if improved:
        best_monitor_score = monitor_score
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(BEST_CHECKPOINT_PATH, epoch, best_monitor_score, history)
        checkpoint_msg = "best saved"
    else:
        epochs_without_improvement += 1
        checkpoint_msg = f"no improve {epochs_without_improvement}/{PATIENCE}"

    save_checkpoint(LAST_CHECKPOINT_PATH, epoch, best_monitor_score, history)
    if epoch % SAVE_EVERY_EPOCHS == 0:
        save_checkpoint(M1_MODEL_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, best_monitor_score, history)

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['loss']:.4f} valid_loss={valid_metrics['loss']:.4f} | "
        f"train_c={train_metrics['c_index']:.3f} valid_c={valid_metrics['c_index']:.3f} | "
        f"valid_AUROC={valid_metrics.get('mean_auroc', float('nan')):.3f} "
        f"valid_AUPRC={valid_metrics.get('mean_auprc', float('nan')):.3f} | "
        f"lr={current_lr:.2e} | {MONITOR_METRIC}={monitor_score:.3f} | best_epoch={best_epoch} ({checkpoint_msg})"
    )

    if epoch % LOG_EVERY_EPOCHS == 0 or epoch == 1:
        plot_training_history(history)

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: {PATIENCE} epochs without monitor improvement.")
        break

print("Training finished.")
print("best_monitor_score:", best_monitor_score)
print("best_epoch:", best_epoch)
print("best_checkpoint:", BEST_CHECKPOINT_PATH)
print("last_checkpoint:", LAST_CHECKPOINT_PATH)
